# Course: COMP 333 — Project Phase 2: Advanced Modeling

**Team D**:
- Ronnie Chan (27206003)
- Patrice Gallant (40301020)
- Nesrine Larbi (40079009)

### **Task Description**
Phase 2 involves:
1. Engineer new features and perform feature selection for the ML models.
2. Implement two supervised ML models (XGBoost and Random Forest) for classification (RQ1).
3. Implement K-means clustering unsupervised ML model and evaluate its quality (RQ2).
4. Interpret results and draw conclusions.

### **Division-of-Labour Statement**
| Student | Tasks |
| :--- | :--- |
| **Ronnie Chan** | |
| **Patrice Gallant** | |
| **Nesrine Larbi** | |

## 1. Import Libraries

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier as randForest
from sklearn.cluster import KMeans, MiniBatchKMeans

from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, silhouette_score,
    confusion_matrix, f1_score, classification_report,
    roc_curve, davies_bouldin_score, calinski_harabasz_score
)
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np
import gc
import time

Load the cleaned dataset produced by Phase 1.

In [ ]:
taxi_df = pd.read_parquet('taxi_cleaned.parquet')

## 2. Feature Engineering
____

### 2.1. Time-Based & Log Features

| Feature | Description |
|---------|-------------|
| `is_high_tip` | **Target for RQ1** — `True` if tip ≥ 20% of base fare (`fare_amount`) |
| `log_trip_distance` | Log-transform of distance to reduce right-skew |
| `log_fare_per_km` | Log-scaled fare efficiency |
| `tip_ratio` | Log-scaled tip-to-fare ratio *(excluded from feature selection — leakage risk)* |
| `hour_sin/cos` | Cyclical encoding of hour (23 h stays close to 0 h) |
| `day_sin/cos` | Cyclical encoding of day of week |

In [ ]:
# Create a boolean is_high_tip feature
taxi_df['is_high_tip'] = taxi_df['tip_amount'] >= taxi_df['fare_amount'] * 0.2

# Fare-based and trip-based features independent of raw scale
taxi_df['log_fare_per_km']   = np.log(np.maximum(taxi_df['fare_amount'], 0) / np.maximum(taxi_df['trip_distance'], 0.5) + 1)
taxi_df['tip_ratio']         = np.log(np.maximum(taxi_df['tip_amount'], 0) / np.maximum(taxi_df['fare_amount'], 1) + 1)
taxi_df['log_trip_distance'] = np.log(taxi_df['trip_distance'] + 1)

# Ensure hour_of_day exists
if 'hour_of_day' not in taxi_df.columns:
    taxi_df['hour_of_day'] = taxi_df['pickup_hour'].dt.hour.astype('int8')

# Cyclical time encodings
taxi_df['hour_sin'] = np.sin(2 * np.pi * taxi_df['hour_of_day'] / 24)
taxi_df['hour_cos'] = np.cos(2 * np.pi * taxi_df['hour_of_day'] / 24)
taxi_df['day_sin']  = np.sin(2 * np.pi * taxi_df['day_of_week'] / 7)
taxi_df['day_cos']  = np.cos(2 * np.pi * taxi_df['day_of_week'] / 7)

### 2.2. Domain-Specific Features

Features derived from NYC taxi domain knowledge:

| Feature | Motivation |
|---------|------------|
| `is_rush_hour` | Weekday 7–9 AM or 5–7 PM — high demand may affect tipping |
| `is_weekend` | Saturday / Sunday ride patterns differ from weekdays |
| `is_night` | 10 PM – 5 AM — night surcharges apply |
| `is_airport` | JFK or Newark flat-rate trip |
| `fare_per_mile` | Cost efficiency of the trip (capped at 99th percentile) |
| `is_credit_card` | Cash tips are **not digitally recorded** → structural zero for cash payments |

In [ ]:
# 1. Rush hour: weekday 7–9 AM or 5–7 PM
taxi_df['is_rush_hour'] = (
    (taxi_df['hour_of_day'].between(7, 9) | taxi_df['hour_of_day'].between(17, 19)) &
    (taxi_df['day_of_week'] < 5)
).astype('int8')

# 2. Weekend
taxi_df['is_weekend'] = (taxi_df['day_of_week'] >= 5).astype('int8')

# 3. Night ride (10 PM – 5 AM)
taxi_df['is_night'] = (
    (taxi_df['hour_of_day'] >= 22) | (taxi_df['hour_of_day'] <= 5)
).astype('int8')

# 4. Airport trip (JFK or Newark flat-rate)
taxi_df['is_airport'] = taxi_df['rate_code'].isin(['JFK', 'Newark']).astype('int8')

# 5. Fare per mile — cap at 99th percentile of positive-fare trips
_cap = (
    taxi_df.loc[taxi_df['fare_amount'] > 0, 'fare_amount'] /
    taxi_df.loc[taxi_df['fare_amount'] > 0, 'trip_distance']
).quantile(0.99)
taxi_df['fare_per_mile'] = (
    (taxi_df['fare_amount'] / taxi_df['trip_distance']).clip(lower=0, upper=_cap)
).astype('float32')
del _cap

# 6. Credit card flag
taxi_df['is_credit_card'] = (taxi_df['payment_type'] == 'Credit card').astype('int8')

print('Domain features added:')
print(taxi_df[['is_rush_hour','is_weekend','is_night',
               'is_airport','fare_per_mile','is_credit_card']].describe().round(3))

### 2.3. Polynomial & Interaction Features

| Feature | Description |
|--------|-------------|
| **trip_distance_sq** | Distance² — fare grows non-linearly on very long trips |
| **distance_x_passengers** | Distance × passenger count — group trips cost more overall |
| **temp_x_precipitation** | Temperature × precipitation — combined bad-weather effect on tipping |

In [ ]:
taxi_df['trip_distance_sq']      = (taxi_df['trip_distance'] ** 2).astype('float32')
taxi_df['distance_x_passengers'] = (taxi_df['trip_distance'] * taxi_df['passenger_count']).astype('float32')
taxi_df['temp_x_precipitation']  = (taxi_df['temperature_2m'] * taxi_df['precipitation']).astype('float32')

print(f'Polynomial/interaction features added.')
print(f'taxi_df now has {taxi_df.shape[1]} columns total.')

### 2.4. Feature Selection

We apply three complementary approaches to find the best features for predicting `is_high_tip`:

1. **Filter methods** — score features without training a model
   - Variance threshold: drop near-constant features
   - Pearson correlation: linear relationship with the target
   - Mutual information: captures non-linear dependencies
2. **Embedded method** — importance scores from a trained Random Forest
3. **Wrapper method** — Recursive Feature Elimination (RFE) with Logistic Regression

> **Leakage note:** `tip_amount`, `total_amount`, and `tip_ratio` are excluded as they are derived from the same quantity as the target `is_high_tip`.

#### 2.4.1. Setup Feature Matrix & Target

In [ ]:
# All engineered features — leaky columns excluded
FEATURE_COLS = [
    # Raw numerical
    'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tolls_amount',
    'improvement_surcharge', 'congestion_surcharge', 'Airport_fee',
    'cbd_congestion_fee', 'passenger_count', 'temperature_2m', 'precipitation',
    'hour_of_day', 'day_of_week', 'day',
    # Log / ratio
    'log_trip_distance', 'log_fare_per_km', 'fare_per_mile',
    # Cyclical time
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    # Domain-specific
    'is_rush_hour', 'is_weekend', 'is_night', 'is_airport', 'is_credit_card',
    # Polynomial & interaction
    'trip_distance_sq', 'distance_x_passengers', 'temp_x_precipitation',
]
TARGET = 'is_high_tip'

# Use only positive-fare trips; drop any remaining NaNs
feat_df = taxi_df[taxi_df['is_negative'] == False][FEATURE_COLS + [TARGET]].dropna()

# 50 000-row sample keeps selection fast
sample = feat_df.sample(n=50_000, random_state=42)
X_all  = sample[FEATURE_COLS].values.astype('float32')
y      = sample[TARGET].astype(int).values

print(f'Feature matrix : {X_all.shape[0]:,} samples x {X_all.shape[1]} features')
print(f'Class balance  : is_high_tip = 1  ->  {y.mean()*100:.1f} %')

#### 2.4.2. Filter Methods

In [ ]:
# ── Variance Threshold ──────────────────────────────────────────────────────
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_all)
removed_low_var = [f for f, keep in zip(FEATURE_COLS, vt.get_support()) if not keep]
print(f'Low-variance features removed: {removed_low_var if removed_low_var else "none"}')

# ── Pearson Correlation with Target ──────────────────────────────────────────
corr_target = (
    sample[FEATURE_COLS]
    .corrwith(sample[TARGET].astype(float))
    .abs()
    .fillna(0)
    .sort_values(ascending=False)
)

# ── Mutual Information ────────────────────────────────────────────────────────
mi_scores = mutual_info_classif(X_all, y, random_state=42)
mi_series = pd.Series(mi_scores, index=FEATURE_COLS).sort_values(ascending=False)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Filter Methods — Feature Ranking', fontsize=14)

mi_series.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].invert_yaxis()
axes[0].set_title('Mutual Information Score')
axes[0].set_xlabel('MI Score')

corr_target.plot(kind='barh', ax=axes[1], color='coral')
axes[1].invert_yaxis()
axes[1].set_title('|Pearson Correlation| with is_high_tip')
axes[1].set_xlabel('|Correlation|')

plt.tight_layout()
plt.show()

print('Top 10 — Mutual Information:')
print(mi_series.head(10).round(4).to_string())
print('\nTop 10 — Pearson Correlation:')
print(corr_target.head(10).round(4).to_string())

#### 2.4.3. Embedded Method — Random Forest Feature Importance

> **Note:** The remainder of this notebook (sections 2.4.3 onwards through the modelling and clustering sections) was not included in the submitted content. Add the remaining cells here.

In [ ]:
# ── Random Forest Feature Importance ────────────────────────────────────────
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_all, y)

rf_importance = pd.Series(rf_selector.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
rf_importance.plot(kind='barh', color='forestgreen')
plt.gca().invert_yaxis()
plt.title('Random Forest — Feature Importance')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 10 — Random Forest Importance:')
print(rf_importance.head(10).round(4).to_string())

#### 2.4.4. Wrapper Method — RFE with Logistic Regression

> Add RFE analysis cells here.

### 2.5. Final Feature Set

> Add final feature selection summary and rationale here.

---

## 3. Supervised Learning (RQ1)

> Add XGBoost and Random Forest classification model cells here.

---

## 4. Unsupervised Learning (RQ2)

> Add K-Means clustering model cells here.

---

## 5. Conclusions

> Add interpretation and conclusions here.